In [ ]:
import pandas as pd
from pathlib import Path

def create_summary_table(csv_path: Path, decimals: int = 3) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    
    # Falls mehrere Runs für dieselbe Kombination aus Kohorte & Modell existieren, 
    # wird automatisch der jeweils neueste (letzte) Eintrag genommen:
    df = df.drop_duplicates(subset=["cohort", "model_type"], keep="last").copy()
    
    # 1. Kohorten-Namen kürzen ("TCGA-KIRC" -> "KIRC")
    df["Dataset"] = df["cohort"].str.replace("TCGA-", "", regex=False)
    
    # 2. Modellnamen auf Paper-Bezeichnungen mappen
    model_names = {
        "abmil": "Attention MIL (WSI Only)",
        "snn": "SNN (Genomic Only)",
        "deepsets": "Deep Sets (WSI Only)",
        "mcat": "MCAT (Multimodal)"
    }
    df["Model"] = df["model_type"].map(lambda x: model_names.get(x, x.upper()))
    
    # 3. C-Index und Standardabweichung als "Mean ± Std" formatieren
    df["Result"] = df.apply(
        lambda r: f"{r['mean_c_index']:.{decimals}f} ± {r['std_c_index']:.{decimals}f}",
        axis=1
    )
    
    # Pivot-Tabelle erzeugen (Zeilen = Modelle, Spalten = Datensätze)
    table = df.pivot(index="Model", columns="Dataset", values="Result")
    
    # Spalten in der gewünschten Reihenfolge sortieren
    col_order = [c for c in ["KIRC", "GBMLGG", "KIRP"] if c in table.columns]
    remaining_cols = [c for c in table.columns if c not in col_order]
    table = table[col_order + remaining_cols]
    
    # Mean über alle Datensätze pro Modell berechnen
    overall_mean = df.groupby("Model")["mean_c_index"].mean()
    table["Overall"] = overall_mean.map(lambda x: f"{x:.{decimals}f}")
    
    table.columns.name = None
    return table.reset_index()

csv_file = Path("./runs/experiment_summary.csv") # ggf. Pfad anpassen
summary_table = create_summary_table(csv_file, decimals=3)

display(summary_table.style.hide(axis="index"))

Model,KIRC,GBMLGG,KIRP,Overall
Attention MIL (WSI Only),0.727 ± 0.058,0.769 ± 0.069,0.796 ± 0.079,0.764
